RQs:
- What is the distribution of contract type between occupations/sectors?
- What is the distribution of salary between occupations/sectors?
    - Avg annual increase by sector?
    - Stretch: hourly pay / annual pay / annualised pay?
- What proportion of ads in each occupation/sector mention learning and development/career progression/CPD?
- What proportion of ads in each occupation/sector mention flexible hours/flexible shifts?

Stretch: unsupervised approach: topic modelling of JQ sentences in early years sector vs a comparison sector

Limitations:
- The sample only covers the years 2021-2023 inclusive

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
from typing import List

from dap_job_quality import PROJECT_DIR, logger
from dap_job_quality.getters.keywords import get_keywords
from dap_job_quality.utils import analysis_utils

In [ ]:
# Produced by pipeline/eyp/stratified_sample.py
afs_raw_sample = pd.read_parquet('s3://open-jobs-lake/job_quality/early_years/evaluation_sample/job_ads_by_sector_region_sample_size_19896.parquet')
# Produced by pipeline/find_job_quality.py
processed_ads = pd.read_parquet('s3://open-jobs-lake/job_quality/outputs/afs/job_ads_prod_True_n_19896.parquet')
# Produced by pipeline/eyp/extract_salaries.py
enhanced_salary_data = pd.read_parquet("s3://open-jobs-lake/job_quality/early_years/evaluation_sample/job_ads_by_sector_region_sample_size_19896_metadata_enhanced_salaries.parquet")

lookup = get_keywords()

In [ ]:
def process_afs_sample(afs_raw_sample):

    category_replacements = {
        'eyp': 'Early Years Practitioner',
        'non education': 'Waiter, Retail Assistant',
        'school teacher': 'Primary/Secondary School Teacher',
        'special needs teacher': 'Special Needs Teacher',
        'supply teacher': 'Supply Teacher',
        'teaching assistant': 'Teaching Assistant',
    }

    geo_replacements = {
        'london': 'London',
        'midlands': 'Midlands',
        'north': 'North',
        'south': 'South'
    }

    # Replace the categories in the column
    afs_raw_sample['sector_group'] = afs_raw_sample['sector_group'].replace(category_replacements)
    afs_raw_sample['geo'] = afs_raw_sample['geo'].replace(geo_replacements)
    afs_raw_sample['sector_group'].value_counts()
    logger.info(f"The complete sample includes {len(afs_raw_sample)} job adverts")
    logger.info(f"The sectors to which EYP roles are compared are: {afs_raw_sample['sector'].unique()}")
    sample_sector_by_itl = afs_raw_sample.groupby(['sector', 'itl_1_name']).agg('size').unstack(fill_value=0)
    logger.info(f'Distribution of sectors by ITL: {sample_sector_by_itl}')
    
    return afs_raw_sample

def process_jq_data(processed_ads):
    """
    Merge the processed job adverts with the lookup table to get the subcategory and dimension of the target phrase.
    
    Create a wide version of the data (dimensions_wide) with one row per job ID.
    """
    processed_ads = processed_ads[['id', 'sentences_split', 'target_phrase']].drop_duplicates()
    processed_ads = pd.merge(processed_ads, lookup[['target_phrase', 'subcategory','dimension']], on='target_phrase', how='left')
    dimensions_wide = analysis_utils.create_wide_table(processed_ads)
    
    return processed_ads, dimensions_wide

def merge_jq_data(afs_raw_sample, dimensions_wide):
    """
    Bring the metadata for the job ads together with the boolean job quality columns
    """
    afs_sample = pd.merge(afs_raw_sample, dimensions_wide, on='id', how='left')
    columns_to_replace = dimensions_wide.columns[2:] # the first column is the id
    # These columns have NaN where there are *no* mentions of JQ dimensions in these job adverts
    afs_sample[columns_to_replace] = afs_sample[columns_to_replace].fillna(0)
    return afs_sample

In [ ]:
afs_raw_sample = process_afs_sample(afs_raw_sample)

In [ ]:
processed_ads, dimensions_wide = process_jq_data(processed_ads)

In [ ]:
afs_sample = merge_jq_data(afs_raw_sample, dimensions_wide)

# Analyse salary data

In [ ]:
afs_sample_enhanced = pd.merge(afs_raw_sample, enhanced_salary_data[['id', 'raw_salary_unit', 'raw_salary_float', 'raw_min_salary_float', 'raw_max_salary_float','is_annualised']], on='id', how='left')

In [ ]:
# Check that the merge was 1-1, and did not introduce duplicates
len(afs_sample_enhanced) - len(afs_raw_sample)

In [ ]:
afs_sample_enhanced['raw_salary_unit'].value_counts()

In [ ]:
# Filter the data to only records that have some kind of salary rate unit
afs_sample_enhanced_w_salaries = afs_sample_enhanced[afs_sample_enhanced['raw_salary_unit'].notna()]
logger.info(f'N adverts lost because they did not contain salary data: {len(afs_sample_enhanced) - len(afs_sample_enhanced_w_salaries)} out of {len(afs_sample_enhanced)} leaving {len(afs_sample_enhanced_w_salaries)}')

In [ ]:
# proportion of ads offering hourly/daily/yearly salary
grouped = afs_sample_enhanced_w_salaries.groupby(['sector_group', 'raw_salary_unit']).size().unstack(fill_value=0)
grouped

In [ ]:
grouped.to_csv('outputs/salary_unit_by_sector.csv')

In [ ]:
afs_sample_enhanced_w_salaries['raw_salary_float'] = afs_sample_enhanced_w_salaries['raw_salary_float'].fillna(afs_sample_enhanced_w_salaries['raw_min_salary_float'])

afs_sample_enhanced_w_salaries['hourly_wage'] = afs_sample_enhanced_w_salaries.apply(analysis_utils.calculate_hourly_wage, axis=1)

In the next chunks we check the distribution of hourly wage and exclude outlying data.

In [ ]:
afs_sample_enhanced_w_salaries['hourly_wage'].describe()

In [ ]:
afs_sample_enhanced_w_salaries['hourly_wage'].hist(bins=100)

In [ ]:
afs_sample_enhanced_w_salaries[afs_sample_enhanced_w_salaries['hourly_wage']<10]['hourly_wage'].hist(bins=100)

In [ ]:
afs_sample_enhanced_w_salaries[afs_sample_enhanced_w_salaries['hourly_wage']>20]['hourly_wage'].hist(bins=100)

In [ ]:
afs_sample_enhanced_w_salaries[(afs_sample_enhanced_w_salaries['hourly_wage']>40) & (afs_sample_enhanced_w_salaries['hourly_wage']<=100)]['sector'].value_counts()

In [ ]:
# Remove outlying salaries from the data.
# The minimum wage for 16 year olds in 2023 was £5.28 so realistically there shouldn't be hourly pay much lower than this.
# Any hourly pay above £40 is also likely to be an error.
afs_sample_enhanced_w_salaries = afs_sample_enhanced_w_salaries[(afs_sample_enhanced_w_salaries['hourly_wage']>=5)&(afs_sample_enhanced_w_salaries['hourly_wage']<40)]

# Salaries by sector and geo

In [ ]:
# Export data for the boxplot of hourly wage by sector
afs_sample_enhanced_w_salaries[['id', 'sector_group', 'hourly_wage']].to_csv('outputs/salary_boxplot.csv')

In [ ]:
afs_sample_enhanced_w_salaries['hourly_wage'].median()

In [ ]:
afs_sample_enhanced_w_salaries.groupby('sector_group')['hourly_wage'].describe()

In [ ]:
geo_median_wage = afs_sample_enhanced_w_salaries.groupby('geo').agg({'hourly_wage': ['median', 'std', 'min', 'max']}).reset_index()
geo_median_wage.to_csv('outputs/geo_median_wage.csv', index=False)
geo_median_wage

In [ ]:
afs_sample_enhanced_w_salaries[['geo', 'sector_group', 'hourly_wage']].to_csv('outputs/geo_sector_hourly_wage_boxplot.csv')

In [ ]:
geo_median_wage_sector = afs_sample_enhanced_w_salaries.groupby(['geo', 'sector_group']).agg({'hourly_wage':'median'}).unstack(fill_value=0).reset_index()
geo_median_wage_sector.to_csv('outputs/geo_median_wage_sector.csv', index=False)
# geo_median_wage_sector
geo_median_wage_sector

# Comparison of multiple dimensions across sectors

In [ ]:
afs_sample_enhanced_w_salaries_dimensions = pd.merge(afs_sample_enhanced_w_salaries, dimensions_wide, on='id', how='left')
len(afs_sample_enhanced_w_salaries_dimensions) - len(afs_sample_enhanced_w_salaries)

In [ ]:
prop_table = afs_sample_enhanced_w_salaries_dimensions.groupby(['sector_group']).agg({'FLEX_HOURS': sum, 'L&D': sum, 'CAREER': sum,'id': 'size', 'hourly_wage': 'median'}).reset_index()

for dim in ['FLEX_HOURS', 'L&D', 'CAREER']:
    prop_table[f'{dim}_perc'] = (prop_table[dim] / prop_table['id']) * 100

prop_table

In [ ]:
prop_table[['sector_group', 'id', 'hourly_wage', 'CAREER_perc', 'FLEX_HOURS_perc', 'L&D_perc']].to_csv('outputs/sector_multi_comparison.csv')

In [ ]:
geo_prop_table = afs_sample_enhanced_w_salaries_dimensions.groupby(['geo','sector_group']).agg({'FLEX_HOURS': sum, 'L&D': sum, 'CAREER': sum,'id': 'size', 'hourly_wage': 'median'}).reset_index()

for dim in ['CAREER', 'FLEX_HOURS', 'L&D']:
    geo_prop_table[f'{dim}_perc'] = (geo_prop_table[dim] / geo_prop_table['id']) * 100

geo_prop_table[['sector_group', 'geo','id', 'hourly_wage', 'CAREER_perc', 'FLEX_HOURS_perc', 'L&D_perc']].to_csv('outputs/sector_geo_multi_comparison.csv')
geo_prop_table


In [ ]:
flex_loc_ids = afs_sample_enhanced_w_salaries_dimensions[(afs_sample_enhanced_w_salaries_dimensions['sector_group']=='Early Years Practitioner') & (afs_sample_enhanced_w_salaries_dimensions['FLEX_LOC']==1)]['id']

In [ ]:
processed_ads[(processed_ads['id'].isin(flex_loc_ids)) & (processed_ads['subcategory']=='FLEX_LOC')]

# Contract type

In [ ]:
contract_refs = processed_ads[processed_ads['subcategory']=='CONTRACT']

In [ ]:
contract_refs.columns

In [ ]:
# Check most frequent words and then we'll do a basic regex for these frequently occurring terms
from wordcloud import WordCloud
import matplotlib.pyplot as plt

text = " ".join(
                contract_refs[
                    "sentences_split"
                ].tolist()
                )

# Generate a word cloud image
wordcloud = WordCloud().generate(text)

# Display the generated image:

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")

In [ ]:
analysis_utils.classify_contract_type('Care Assistant  Guaranteed minimum 20-hour contract Salary  £11per hour weekday evenings and  £12 per hour weekends Hours 5 00 pm to 10 00 pm evenings and alternate weekends Location  Chichester, Bognor Regis and surrounding areas.')

In [ ]:
contract_refs['classified_contract_type'] = contract_refs['sentences_split'].apply(analysis_utils.classify_contract_type)
contract_refs['classified_contract_type'].value_counts()

In [ ]:
#Inspect the ones that couldn't be identified
contract_refs[contract_refs['classified_contract_type']=='Unknown']['sentences_split'].to_list()

In [ ]:
contract_refs['classified_contract_type'].value_counts()

Some IDs get matched to more than one contract type, so we need to pick just one for our output.

In [ ]:
counts = contract_refs.groupby(['id', 'classified_contract_type']).size().reset_index(name='count')

# Find the most frequent 'classified_contract_type' for each job
most_frequent = counts.loc[counts.groupby('id')['count'].idxmax()]

# gather all the different contract types that were applied to the same job
contract_df = contract_refs.groupby('id')['classified_contract_type'].agg(list).reset_index()

def determine_contract_type(types: List[str]):
    if 'Temporary' in types:
        return 'Temporary'
    else:
        # Count occurrences of each type
        type_counts = pd.Series(types).value_counts()
        most_common = type_counts.idxmax()
        if len(type_counts) == 1:  # Only one unique type
            return most_common
        elif len(type_counts) > 1:
            # If the list contains more than one type, we need to check the most common
            if most_common == 'Permanent' or most_common == 'Unknown':
                return most_common
            else:
                return 'Unknown'  # Fallback to 'Unknown' if neither 'Temporary' nor most frequent matches
        return 'Unknown'

# Step 5: Apply the function to determine the final contract type for each group
contract_df['final_contract_type'] = contract_df['classified_contract_type'].apply(determine_contract_type)

In [ ]:
contract_df

In [ ]:
afs_sample_enhanced_contract = pd.merge(afs_sample_enhanced, contract_df[['id', 'final_contract_type']], on='id', how='left')

In [ ]:
afs_sample_enhanced_contract['final_contract_type'].value_counts(dropna=False)

In [ ]:
afs_sample_enhanced_contract['final_contract_type'] = afs_sample_enhanced_contract['final_contract_type'].fillna('Unknown')

In [ ]:
afs_sample_enhanced_contract['final_contract_type'].value_counts(dropna=False)

In [ ]:
afs_sample_enhanced_contract.groupby(['sector_group', 'final_contract_type']).size()

In [ ]:
contract_prop_df = afs_sample_enhanced_contract.groupby(['sector_group', 'final_contract_type']).size().unstack(fill_value=0)
contract_prop_df

In [ ]:
contract_prop_df.to_csv('outputs/sector_contract_type.csv')

# Extracting hours

Not in the final analysis.

In [ ]:
hours_refs = processed_ads[processed_ads['subcategory']=='HOURS']
hours_refs

In [ ]:
hours_refs['hours_per_week'] = hours_refs['sentences_split'].apply(analysis_utils.match_hours_per_week)
hours_refs['working_days'] = hours_refs['sentences_split'].apply(analysis_utils.count_working_days)
hours_refs['is_full_time'] = hours_refs['sentences_split'].apply(analysis_utils.check_full_time)
hours_refs['hr_per_week_final'] = hours_refs.apply(analysis_utils.calculate_hr_per_week_final, axis=1)
hours_refs

In [ ]:
hours_refs['hr_per_week_final'].value_counts(dropna=False)